# Global Player Mobility in Football

## Analysis (from prepared edge lists)

In [15]:
from pathlib import Path
import pandas as pd
import igraph as ig
import leidenalg
import numpy as np

PROJECT_ROOT = Path.cwd()

# Default analysis input
EDGE_ALL_PATH = PROJECT_ROOT / 'data' / 'prepared' / 'edge_all_strict.csv'
EDGE_SEASON_PATH = PROJECT_ROOT / 'data' / 'prepared' / 'edge_season_strict.csv'

# Alternative (with unknown competitions kept)
# EDGE_ALL_PATH = PROJECT_ROOT / 'data' / 'prepared' / 'edge_all_with_unknown.csv'
# EDGE_SEASON_PATH = PROJECT_ROOT / 'data' / 'prepared' / 'edge_season_with_unknown.csv'

edge_all = pd.read_csv(EDGE_ALL_PATH)
edge_season = pd.read_csv(EDGE_SEASON_PATH)

pd.DataFrame({
    'table': ['edge_all', 'edge_season'],
    'rows': [len(edge_all), len(edge_season)],
    'cols': [edge_all.shape[1], edge_season.shape[1]],
})

,table,rows,cols
0,edge_all,31382,13
1,edge_season,115229,14


## 05b  Graph Construction
Build directed graphs from the edge lists

In [2]:
src_labels = edge_all[['source_league_id', 'source_league']].dropna().drop_duplicates()
dst_labels = edge_all[['dest_league_id', 'dest_league']].dropna().drop_duplicates()
src_labels.columns = ['league_id', 'league_name']
dst_labels.columns = ['league_id', 'league_name']
league_labels = (
    pd.concat([src_labels, dst_labels], ignore_index=True)
    .drop_duplicates(subset=['league_id'])
    .set_index('league_id')['league_name']
    .to_dict()
)

agg_tuples = [(r.source_league_id, r.dest_league_id, float(r.n_transfers)) for r in edge_all.itertuples(index=False)]
G_all = ig.Graph.TupleList(agg_tuples, directed=True, weights=True)
G_all.es['distance'] = [1.0 / w if w > 0 else 1e9 for w in G_all.es['weight']]
G_all.vs['league_name'] = [league_labels.get(v['name'], v['name']) for v in G_all.vs]

graphs_by_season = {}
for season, df in edge_season.groupby('season'):
    tuples = [(r.source_league_id, r.dest_league_id, float(r.n_transfers)) for r in df.itertuples(index=False)]
    g = ig.Graph.TupleList(tuples, directed=True, weights=True)
    g.es['distance'] = [1.0 / w if w > 0 else 1e9 for w in g.es['weight']]
    g.vs['league_name'] = [league_labels.get(v['name'], v['name']) for v in g.vs]
    graphs_by_season[season] = g

G_all.vcount(), G_all.ecount()

(650, 31382)

## 6 Network Metrics 

## 6a Basic Metrics: weighted in/out strength, degree, and betweenness for the aggregated graph

In [3]:
metrics = pd.DataFrame({
    'league': G_all.vs['name'],
    'league_name': G_all.vs['league_name'],
    'in_strength': G_all.strength(mode='in', weights='weight'),
    'out_strength': G_all.strength(mode='out', weights='weight'),
    'in_degree': G_all.degree(mode='in'),
    'out_degree': G_all.degree(mode='out'),
    # Betweenness uses distance, so we invert transfer weights in graph construction
    'betweenness': G_all.betweenness(directed=True, weights='distance'),
})

metrics.sort_values('in_strength', ascending=False).head(15)

,league,league_name,in_strength,out_strength,in_degree,out_degree,betweenness
20,WITHOUT_CLUB,WITHOUT_CLUB,45886.0,33456.0,583,576,383994.166667
91,RETIREMENT,RETIREMENT,16905.0,58.0,476,32,5607.000000
135,IT1,Serie A,16002.0,16600.0,193,172,652.000000
84,IT2,Serie B,12539.0,13105.0,170,153,1841.000000
196,GB2,Championship,11429.0,12195.0,183,174,4164.000000
174,BRA1,Campeonato Brasileiro Série A,10988.0,12437.0,164,181,9266.000000
31,GB3,League One,9479.0,9939.0,129,139,8646.000000
94,TR1,Süper Lig,8756.0,9255.0,179,162,581.000000
195,GB1,Premier League,8698.0,8987.0,140,121,1223.000000
175,BRA2,Campeonato Brasileiro Série B,8594.0,8823.0,166,173,29238.000000


## 06b Communities
We will use Leidenalg here for community detection

In [4]:
parts = leidenalg.find_partition(
    G_all,
    leidenalg.RBConfigurationVertexPartition,
    weights='weight',
    seed=42,
)

communities = {G_all.vs[idx]['name']: int(parts.membership[idx]) for idx in range(G_all.vcount())}
metrics['community'] = metrics['league'].map(communities)
'leiden'

'leiden'

## 06c  Pseudo-League Filter & Community Summary

`WITHOUT_CLUB`, `RETIREMENT`, and `UNKNOWN_CLUB` are not real football leagues — they represent players leaving football or going between clubs without a competition. They dominate strength and betweenness rankings artificially and must be excluded from all downstream analysis.

In [ ]:
PSEUDO_LEAGUES = {'WITHOUT_CLUB', 'RETIREMENT', 'UNKNOWN_CLUB'}

metrics_clean = metrics[~metrics['league'].isin(PSEUDO_LEAGUES)].copy().reset_index(drop=True)
edge_all_clean = edge_all[
    ~edge_all['source_league_id'].isin(PSEUDO_LEAGUES) &
    ~edge_all['dest_league_id'].isin(PSEUDO_LEAGUES)
].copy()

print(f"Leagues  : {len(metrics_clean):4d}  (removed {len(metrics) - len(metrics_clean)} pseudo-nodes)")
print(f"Edges    : {len(edge_all_clean):5d}  (removed {len(edge_all) - len(edge_all_clean)} edges)")

In [ ]:
# Community summary (on clean metrics)
comm_summary = (
    metrics_clean
    .groupby('community')
    .apply(lambda g: pd.Series({
        'n_leagues': len(g),
        'total_in_strength': g['in_strength'].sum(),
        'top_league': g.nlargest(1, 'in_strength')['league_name'].values[0],
    }))
    .sort_values('total_in_strength', ascending=False)
    .reset_index()
)
print(f"Number of communities detected: {len(comm_summary)}")
comm_summary.head(15)

## 07 Hypothesis Tests & Robustness

## 7a - H1 – Concentration: Incoming transfer flows are highly concentrated, with a small set of leagues accounting for a large share of all incoming transfers.

In [ ]:
inflow = (edge_all.groupby('dest_league_id', dropna=False)
    .agg(total_in_transfers=('n_transfers','sum'))
    .reset_index()
    .sort_values('total_in_transfers', ascending=False))

name_map = metrics.set_index('league')['league_name'].to_dict()
inflow['league_name'] = inflow['dest_league_id'].map(name_map).fillna(inflow['dest_league_id'])

inflow.head(20)

,dest_league_id,total_in_transfers,league_name
626,WITHOUT_CLUB,45886,WITHOUT_CLUB
496,RETIREMENT,16905,RETIREMENT
292,IT1,16002,Serie A
293,IT2,12539,Serie B
246,GB2,11429,Championship
84,BRA1,10988,Campeonato Brasileiro Série A
248,GB3,9479,League One
569,TR1,8756,Süper Lig
244,GB1,8698,Premier League
85,BRA2,8594,Campeonato Brasileiro Série B


In [8]:
total_in = inflow['total_in_transfers'].sum()
top5_share = inflow.head(5)['total_in_transfers'].sum() / total_in if total_in > 0 else 0
top10_share = inflow.head(10)['total_in_transfers'].sum() / total_in if total_in > 0 else 0
shares = inflow['total_in_transfers'] / total_in if total_in > 0 else inflow['total_in_transfers']
hhi = (shares ** 2).sum()

pd.DataFrame({
    'total_in_transfers': [total_in],
    'top5_share': [top5_share],
    'top10_share': [top10_share],
    'hhi': [hhi],
}).round(4)


,total_in_transfers,top5_share,top10_share,hhi
0,586520,0.1752,0.2545,0.014


### H1 – Clean data (pseudo-leagues excluded) + Robustness check

The numbers above include WITHOUT_CLUB (rank 1) and RETIREMENT (rank 2). Below is the corrected analysis and a robustness check excluding loan transfers.

In [ ]:
# H1 – Concentration on real leagues only
inflow_clean = (
    edge_all_clean
    .groupby('dest_league_id', dropna=False)
    .agg(total_in_transfers=('n_transfers', 'sum'))
    .reset_index()
    .sort_values('total_in_transfers', ascending=False)
)
inflow_clean['league_name'] = inflow_clean['dest_league_id'].map(name_map).fillna(inflow_clean['dest_league_id'])

total_in_c = inflow_clean['total_in_transfers'].sum()
top5_c  = inflow_clean.head(5)['total_in_transfers'].sum()  / total_in_c
top10_c = inflow_clean.head(10)['total_in_transfers'].sum() / total_in_c
shares_c = inflow_clean['total_in_transfers'] / total_in_c
hhi_c = (shares_c ** 2).sum()

# Robustness: permanent transfers only (exclude loans)
edge_all_clean['n_permanent'] = edge_all_clean['n_transfers'] - edge_all_clean['n_loans']
edge_perm = edge_all_clean[edge_all_clean['n_permanent'] > 0].copy()

inflow_perm = (
    edge_perm
    .groupby('dest_league_id')
    .agg(total_in_transfers=('n_permanent', 'sum'))
    .reset_index()
    .sort_values('total_in_transfers', ascending=False)
)
total_p = inflow_perm['total_in_transfers'].sum()
top5_p  = inflow_perm.head(5)['total_in_transfers'].sum()  / total_p
top10_p = inflow_perm.head(10)['total_in_transfers'].sum() / total_p
hhi_p   = ((inflow_perm['total_in_transfers'] / total_p) ** 2).sum()

pd.DataFrame({
    'metric': ['top5_share', 'top10_share', 'hhi', 'n_leagues'],
    'all_transfers (clean)': [round(top5_c, 4), round(top10_c, 4), round(hhi_c, 4), len(inflow_clean)],
    'permanent_only':        [round(top5_p, 4), round(top10_p, 4), round(hhi_p, 4), len(inflow_perm)],
})

## 7b - H2 – Bridges: Some mid-level leagues exhibit disproportionately high betweenness centrality and act as bridges between regions and/or tier systems.

In [10]:
tier_map_src = edge_all[['source_league_id','source_tier']].dropna().drop_duplicates().rename(columns={'source_league_id':'league','source_tier':'tier'})
tier_map_dst = edge_all[['dest_league_id','dest_tier']].dropna().drop_duplicates().rename(columns={'dest_league_id':'league','dest_tier':'tier'})
tier_map = pd.concat([tier_map_src, tier_map_dst], ignore_index=True).drop_duplicates(subset=['league'])
tier_map_dict = tier_map.set_index('league')['tier'].to_dict()

metrics['tier'] = metrics['league'].map(tier_map_dict).fillna('Unknown')
metrics['tier_num'] = pd.to_numeric(metrics['tier'], errors='coerce')
metrics['is_mid_tier'] = metrics['tier_num'].between(2, 4, inclusive='both').fillna(False)

metrics[['league','league_name','tier','is_mid_tier','betweenness']].head()

,league,league_name,tier,is_mid_tier,betweenness
0,17LA,U17 DFB-Nachwuchsliga - Hauptrunde Liga A,Youth,False,0.0
1,19LA,U19 DFB-Nachwuchsliga - Hauptrunde Liga A,Youth,False,623.0
2,17LB,U17 DFB-Nachwuchsliga - Hauptrunde Liga B,Youth,False,0.0
3,19LB,U19 DFB-Nachwuchsliga - Hauptrunde Liga B,Youth,False,624.0
4,181F,U18 Divisie 1 Herbst,Youth,False,1278.0


In [11]:
bridge_rank = (metrics
    .sort_values('betweenness', ascending=False)
    [['league','league_name','tier','is_mid_tier','betweenness','in_strength','out_strength']])
bridge_rank.head(25)

,league,league_name,tier,is_mid_tier,betweenness,in_strength,out_strength
20,WITHOUT_CLUB,WITHOUT_CLUB,Unknown,False,383994.166667,45886.0,33456.0
175,BRA2,Campeonato Brasileiro Série B,2,True,29238.000000,8594.0,8823.0
155,A2,2. Liga,2,True,27094.000000,2708.0,2802.0
336,ARG2,Primera Nacional,2,True,21373.500000,5424.0,5734.0
215,L3,3. Liga,3,True,17090.666667,4786.0,4952.0
15,NL2,Keuken Kampioen Divisie,2,True,16786.000000,4165.0,4640.0
53,RLW3,Regionalliga West,4,True,14977.666667,2326.0,2579.0
77,C2,Challenge League,2,True,14072.000000,3013.0,3122.0
121,RU2,1.Division,2,True,13751.000000,5767.0,5925.0
368,MEX2,Liga de Expansión MX Clausura,2,True,13510.000000,4155.0,4211.0


In [12]:
k = 25
topk = bridge_rank.head(k)
mid_share_topk = topk['is_mid_tier'].mean() if len(topk) else 0
mid_share_all = metrics['is_mid_tier'].mean() if len(metrics) else 0

pd.DataFrame({
    'k': [k],
    'mid_share_topk_betweenness': [mid_share_topk],
    'mid_share_all_leagues': [mid_share_all],
    'lift_topk_vs_all': [mid_share_topk / mid_share_all if mid_share_all > 0 else None],
}).round(4)

,k,mid_share_topk_betweenness,mid_share_all_leagues,lift_topk_vs_all
0,25,0.8,0.3815,2.0968


## 07d - H4 – Lower-League Pathways

**Hypothesis:** Transfers from lower tiers are more likely to be domestic and upward (lower → higher tier within the same country), while cross-border moves are relatively less common compared to top-tier mobility.

In [ ]:
# H4 – Prepare tier-annotated edge frame (real leagues only, both tiers numeric)
edge_tiered = edge_all_clean.copy()
edge_tiered['source_tier_num'] = pd.to_numeric(edge_tiered['source_tier'], errors='coerce')
edge_tiered['dest_tier_num']   = pd.to_numeric(edge_tiered['dest_tier'],   errors='coerce')
edge_tiered = edge_tiered.dropna(subset=['source_tier_num', 'dest_tier_num'])
edge_tiered['source_tier_num'] = edge_tiered['source_tier_num'].astype(int)
edge_tiered['dest_tier_num']   = edge_tiered['dest_tier_num'].astype(int)

print(f"Edges with numeric tiers on both sides: {len(edge_tiered)} / {len(edge_all_clean)}")

# Tier-flow matrix: total transfers by (source_tier, dest_tier)
tier_matrix = (
    edge_tiered
    .groupby(['source_tier_num', 'dest_tier_num'])
    .agg(n_transfers=('n_transfers', 'sum'))
    .reset_index()
    .pivot(index='source_tier_num', columns='dest_tier_num', values='n_transfers')
    .fillna(0).astype(int)
)
tier_matrix.index.name   = 'source_tier'
tier_matrix.columns.name = 'dest_tier'
tier_matrix

In [ ]:
# H4 – Domestic share by tier group (weighted by n_transfers)
def weighted_domestic(grp):
    total = grp['n_transfers'].sum()
    return (grp['n_transfers'] * grp['domestic_share']).sum() / total if total > 0 else np.nan

def tier_group(t):
    if t == 1:   return 'Top (Tier 1)'
    elif t == 2: return 'Second (Tier 2)'
    elif t <= 4: return 'Mid (Tier 3-4)'
    else:        return 'Lower (Tier 5+)'

edge_tiered['tier_group'] = edge_tiered['source_tier_num'].apply(tier_group)
group_order = ['Top (Tier 1)', 'Second (Tier 2)', 'Mid (Tier 3-4)', 'Lower (Tier 5+)']

h4_domestic = (
    edge_tiered
    .groupby('tier_group')
    .apply(lambda g: pd.Series({
        'n_transfers':         g['n_transfers'].sum(),
        'n_source_leagues':    g['source_league_id'].nunique(),
        'domestic_share':      round(weighted_domestic(g), 4),
        'international_share': round(1 - weighted_domestic(g), 4),
    }))
    .reindex(group_order)
    .reset_index()
)
h4_domestic

In [ ]:
# H4 – Directional flow (upward / same / downward) by tier group
edge_tiered['is_upward']    = edge_tiered['dest_tier_num'] < edge_tiered['source_tier_num']
edge_tiered['is_same_tier'] = edge_tiered['dest_tier_num'] == edge_tiered['source_tier_num']
edge_tiered['is_downward']  = edge_tiered['dest_tier_num'] > edge_tiered['source_tier_num']

h4_direction = (
    edge_tiered
    .groupby('tier_group')
    .apply(lambda g: pd.Series({
        'n_transfers':      g['n_transfers'].sum(),
        'upward_share':     round((g['n_transfers'] * g['is_upward']).sum()    / g['n_transfers'].sum(), 4),
        'same_tier_share':  round((g['n_transfers'] * g['is_same_tier']).sum() / g['n_transfers'].sum(), 4),
        'downward_share':   round((g['n_transfers'] * g['is_downward']).sum()  / g['n_transfers'].sum(), 4),
    }))
    .reindex(group_order)
    .reset_index()
)
h4_direction

## 07c - H3 – Communities: The network exhibits non-random community structure, with detected communities showing higher modularity and/or greater stability than in a randomized baseline network.

In [13]:
observed_modularity = parts.quality()
observed_modularity

351360.49653208756

In [ ]:
import gc

# H3 – Fix: use ModularityVertexPartition (gives standard Q, scale-independent)
# so that observed and null are directly comparable.
# Also remove self-loops which break degree-preserving rewire.

G_nsl = G_all.copy()
G_nsl.simplify(loops=True, multiple=False)   # remove self-loops & multi-edges
print(f"Edges for null model: {G_nsl.ecount()} (G_all had {G_all.ecount()})")

obs_parts_Q = leidenalg.find_partition(
    G_nsl,
    leidenalg.ModularityVertexPartition,
    seed=42,
)
observed_Q = obs_parts_Q.quality()
print(f"Observed Q (ModularityVertexPartition, no self-loops): {observed_Q:.6f}")

n_runs       = 20
niter_factor = 2   # reduced from 10 to avoid memory exhaustion
null_Q       = []

for run in range(n_runs):
    g_null   = G_nsl.copy()
    n_rewire = int(max(1, g_null.ecount() * niter_factor))
    try:
        g_null.rewire(n=n_rewire, allowed_edge_types='simple')
    except TypeError:
        g_null.rewire(n=n_rewire, mode='simple')
    null_parts = leidenalg.find_partition(
        g_null,
        leidenalg.ModularityVertexPartition,
        seed=42 + run,
    )
    null_Q.append(null_parts.quality())
    del g_null, null_parts
    gc.collect()

null_Q = np.array(null_Q)
pd.Series(null_Q).describe()

In [ ]:
null_mean_Q = float(null_Q.mean())
null_std_Q  = float(null_Q.std(ddof=1)) if len(null_Q) > 1 else np.nan
z_score_Q   = (observed_Q - null_mean_Q) / null_std_Q if null_std_Q and not np.isnan(null_std_Q) else np.nan
p_val_Q     = float((null_Q >= observed_Q).mean())   # one-sided empirical p-value

pd.DataFrame({
    'observed_Q':        [observed_Q],
    'null_mean_Q':       [null_mean_Q],
    'null_std_Q':        [null_std_Q],
    'z_score':           [z_score_Q],
    'p_value_one_sided': [p_val_Q],
}).round(6)

## 08 Visuals 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

os.makedirs('data/prepared', exist_ok=True)

# ── Visual 1: Top-20 league inflow bar chart ─────────────────────────────────
top20 = inflow_clean.head(20).copy()
top20['short'] = top20['league_name'].str.slice(0, 22)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top20['short'][::-1], top20['total_in_transfers'][::-1], color='steelblue')
ax.set_xlabel('Total incoming transfers (aggregated)')
ax.set_title('Top 20 Leagues by Incoming Transfers\n(pseudo-leagues excluded)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('data/prepared/fig1_top20_inflow.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: data/prepared/fig1_top20_inflow.png")

In [ ]:
# ── Visual 2: League-to-League Flow Heatmap (top 15 x top 15) ────────────────
top15_ids = inflow_clean.head(15)['dest_league_id'].tolist()

heat = (
    edge_all_clean[
        edge_all_clean['source_league_id'].isin(top15_ids) &
        edge_all_clean['dest_league_id'].isin(top15_ids)
    ]
    .groupby(['source_league_id', 'dest_league_id'])
    .agg(n=('n_transfers', 'sum'))
    .reset_index()
    .pivot(index='source_league_id', columns='dest_league_id', values='n')
    .reindex(index=top15_ids, columns=top15_ids)
    .fillna(0)
)
lname = metrics_clean.set_index('league')['league_name'].to_dict()
heat.index   = [lname.get(l, l)[:18] for l in heat.index]
heat.columns = [lname.get(l, l)[:18] for l in heat.columns]

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(heat.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels(heat.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(heat.index)));   ax.set_yticklabels(heat.index, fontsize=8)
ax.set_title('League-to-League Transfer Flow Heatmap\n(Top 15 leagues by inflow, self-loops on diagonal)', fontsize=12)
ax.set_xlabel('Destination league'); ax.set_ylabel('Source league')
plt.colorbar(im, ax=ax, label='Number of transfers')
plt.tight_layout()
plt.savefig('data/prepared/fig2_heatmap_top15.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: data/prepared/fig2_heatmap_top15.png")

In [ ]:
# ── Visual 3: Community sizes + dominant leagues ──────────────────────────────
top_comms = comm_summary.head(10).copy()
top_comms['label'] = top_comms.apply(
    lambda r: f"C{int(r['community'])}: {r['top_league'][:20]} (+{int(r['n_leagues'])-1} more)", axis=1
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_comms['label'][::-1], top_comms['total_in_strength'][::-1], color='darkorange')
ax.set_xlabel('Total in-strength (sum of incoming transfers)')
ax.set_title('Top 10 Communities by Transfer Volume\n(Leiden algorithm, aggregated graph)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('data/prepared/fig3_community_sizes.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: data/prepared/fig3_community_sizes.png")

In [ ]:
# ── Visual 4: H4 – Domestic vs International share by tier group ──────────────
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(group_order))
w = 0.35
ax.bar(x - w/2, h4_domestic['domestic_share'],      w, label='Domestic',      color='steelblue')
ax.bar(x + w/2, h4_domestic['international_share'],  w, label='International', color='tomato')
ax.set_xticks(x); ax.set_xticklabels(group_order, fontsize=10)
ax.set_ylabel('Share of transfers')
ax.set_title('Domestic vs International Transfer Share by Source Tier', fontsize=12)
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('data/prepared/fig4_h4_tier_domestic.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: data/prepared/fig4_h4_tier_domestic.png")

## 09 Results Summary (stub)

In [ ]:
# ── KPI Summary ───────────────────────────────────────────────────────────────
kpi = []

# H1 – Concentration
kpi.append({
    'Hypothesis': 'H1 – Concentration',
    'Key Metric': f'Top-5 share: {top5_c:.1%} | Top-10: {top10_c:.1%} | HHI: {hhi_c:.4f}',
    'Robustness': f'Perm-only top-10: {top10_p:.1%} | HHI: {hhi_p:.4f}',
    'Verdict': 'Moderate concentration; HHI well below 0.25 → market not monopolistic but unequal',
})

# H2 – Bridges
kpi.append({
    'Hypothesis': 'H2 – Bridges (mid-tier)',
    'Key Metric': f'Mid-tier share in top-{k} betweenness: {mid_share_topk:.1%} vs {mid_share_all:.1%} overall',
    'Robustness': f'Lift = {mid_share_topk/mid_share_all:.2f}x',
    'Verdict': 'Supported — mid-tier leagues (2-4) are ~2x over-represented among high-betweenness nodes',
})

# H3 – Communities
h3_verdict = 'Strongly supported (p < 0.05)' if p_val_Q < 0.05 else 'Not significant (p ≥ 0.05)'
kpi.append({
    'Hypothesis': 'H3 – Non-random community structure',
    'Key Metric': f'Observed Q: {observed_Q:.4f} | Null mean Q: {null_mean_Q:.4f} | z = {z_score_Q:.2f} | p = {p_val_Q:.3f}',
    'Robustness': f'20 degree-preserving rewired null graphs',
    'Verdict': h3_verdict,
})

# H4 – Lower-league pathways
try:
    h4_top    = h4_domestic[h4_domestic['tier_group'] == 'Top (Tier 1)'].iloc[0]
    h4_lower  = h4_domestic[h4_domestic['tier_group'] == 'Lower (Tier 5+)'].iloc[0]
    h4_dir_l  = h4_direction[h4_direction['tier_group'] == 'Lower (Tier 5+)'].iloc[0]
    h4_dir_t  = h4_direction[h4_direction['tier_group'] == 'Top (Tier 1)'].iloc[0]
    kpi.append({
        'Hypothesis': 'H4 – Lower-league pathways',
        'Key Metric': (
            f"Domestic share — Top: {h4_top['domestic_share']:.1%}, Lower 5+: {h4_lower['domestic_share']:.1%} | "
            f"Upward share — Top: {h4_dir_t['upward_share']:.1%}, Lower 5+: {h4_dir_l['upward_share']:.1%}"
        ),
        'Robustness': 'Based on aggregated edge domestic_share column',
        'Verdict': (
            'Supported' if h4_lower['domestic_share'] > h4_top['domestic_share']
            else 'Not supported — lower-tier not more domestic than top-tier'
        ),
    })
except Exception as e:
    kpi.append({'Hypothesis': 'H4', 'Key Metric': str(e), 'Robustness': '', 'Verdict': ''})

kpi_df = pd.DataFrame(kpi)
pd.set_option('display.max_colwidth', 120)
kpi_df